# GDELT Events — Free HTTP Sample

Downloads `SAMPLE_DAYS` days of GDELT 1.0 Events data via direct HTTP (no BigQuery, no credentials, no cost).  
Applies zone + QuadClass filters, writes a Bronze JSONL sample, and reports concrete size metrics  
so we can extrapolate the full 2021→today backfill cost before committing.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SAMPLE_DAYS     = 30     # days to sample, counting back from yesterday
SLEEP_BETWEEN   = 1.5    # seconds between HTTP requests
OUTPUT_DIR      = "/content/gdelt_events_sample"

In [ ]:
import csv
import io
import json
import os
import time
import zipfile
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Zone constants ─────────────────────────────────────────────────────────────
ZONE_COUNTRY_CODES = {
    "USD": ["USA"],
    "EUR": [
        "EUR", "AUT", "BEL", "CYP", "DEU", "ESP", "EST", "FIN", "FRA",
        "GRC", "HRV", "IRL", "ITA", "LTU", "LUX", "LVA", "MLT",
        "NLD", "PRT", "SVK", "SVN",
    ],
    "GBP": ["GBR"],
    "JPY": ["JPN"],
    "CHF": ["CHE"],
}
ALL_ZONE_CODES = {code for codes in ZONE_COUNTRY_CODES.values() for code in codes}

# reverse map: country code -> zone label
COUNTRY_TO_ZONE = {code: zone for zone, codes in ZONE_COUNTRY_CODES.items() for code in codes}

# ── GDELT 1.0 column names (58 columns, tab-separated, no header) ──────────────
ALL_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",
    "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_Lat", "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_Lat", "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_Lat", "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL",
]
assert len(ALL_COLUMNS) == 58, f"Expected 58 columns, got {len(ALL_COLUMNS)}"

In [ ]:
# ── Download + filter one day ──────────────────────────────────────────────────
BASE_URL = "http://data.gdeltproject.org/events/{}.export.CSV.zip"

def _str_or_none(val):
    if val is None:
        return None
    s = str(val).strip()
    return s if s and s.lower() not in {"nan", "none", ""} else None

def _safe_float(val):
    try:
        return float(val)
    except (TypeError, ValueError):
        return None

def _safe_int(val):
    try:
        return int(float(val))
    except (TypeError, ValueError):
        return None


def download_and_filter_day(d: date):
    """Returns (raw_rows, filtered_rows, zip_bytes, records) or None on 404/error."""
    url = BASE_URL.format(d.strftime("%Y%m%d"))
    try:
        resp = requests.get(url, timeout=60)
    except requests.RequestException as exc:
        print(f"  [SKIP] {d} — request error: {exc}")
        return None

    if resp.status_code == 404:
        print(f"  [SKIP] {d} — 404 not found")
        return None

    if resp.status_code != 200:
        print(f"  [SKIP] {d} — HTTP {resp.status_code}")
        return None

    zip_bytes = len(resp.content)

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        csv_name = zf.namelist()[0]
        with zf.open(csv_name) as f:
            df = pd.read_csv(
                f,
                sep="\t",
                header=None,
                names=ALL_COLUMNS,
                dtype=str,
                quoting=csv.QUOTE_NONE,
                on_bad_lines="skip",
            )

    raw_rows = len(df)

    # zone filter
    zone_mask = (
        df["Actor1CountryCode"].isin(ALL_ZONE_CODES)
        | df["Actor2CountryCode"].isin(ALL_ZONE_CODES)
    )
    # QuadClass filter (2=MatCoop, 3=VerConflict, 4=MatConflict)
    quad_mask = df["QuadClass"].isin({"2", "3", "4"})

    filtered = df[zone_mask & quad_mask]

    records = []
    for row in filtered.itertuples(index=False):
        records.append({
            "event_id":               _str_or_none(row.GLOBALEVENTID),
            "event_date":             _str_or_none(row.SQLDATE),
            "actor1_name":            _str_or_none(row.Actor1Name),
            "actor1_code":            _str_or_none(row.Actor1Code),
            "actor1_country_code":    _str_or_none(row.Actor1CountryCode),
            "actor1_type1_code":      _str_or_none(row.Actor1Type1Code),
            "actor2_name":            _str_or_none(row.Actor2Name),
            "actor2_code":            _str_or_none(row.Actor2Code),
            "actor2_country_code":    _str_or_none(row.Actor2CountryCode),
            "actor2_type1_code":      _str_or_none(row.Actor2Type1Code),
            "event_code":             _str_or_none(row.EventCode),
            "event_base_code":        _str_or_none(row.EventBaseCode),
            "event_root_code":        _str_or_none(row.EventRootCode),
            "quad_class":             _safe_int(row.QuadClass),
            "goldstein_scale":        _safe_float(row.GoldsteinScale),
            "num_mentions":           _safe_int(row.NumMentions),
            "num_sources":            _safe_int(row.NumSources),
            "num_articles":           _safe_int(row.NumArticles),
            "avg_tone":               _safe_float(row.AvgTone),
            "actor1_geo_country_code":_str_or_none(row.Actor1Geo_CountryCode),
            "actor1_geo_full_name":   _str_or_none(row.Actor1Geo_FullName),
            "actor2_geo_country_code":_str_or_none(row.Actor2Geo_CountryCode),
            "actor2_geo_full_name":   _str_or_none(row.Actor2Geo_FullName),
            "action_geo_country_code":_str_or_none(row.ActionGeo_CountryCode),
            "action_geo_full_name":   _str_or_none(row.ActionGeo_FullName),
            "source_url":             _str_or_none(row.SOURCEURL),
        })

    return raw_rows, len(records), zip_bytes, records

In [ ]:
# ── Collect SAMPLE_DAYS days ───────────────────────────────────────────────────
yesterday = date.today() - timedelta(days=1)
days = [yesterday - timedelta(days=i) for i in range(SAMPLE_DAYS)]
days = sorted(days)  # oldest first

total_raw_rows      = 0
total_filtered_rows = 0
total_zip_bytes     = 0
total_jsonl_bytes   = 0
days_collected      = 0
days_skipped        = 0

output_path = Path(OUTPUT_DIR) / "sample.jsonl"

with output_path.open("w", encoding="utf-8") as out_f:
    for d in days:
        result = download_and_filter_day(d)
        if result is None:
            days_skipped += 1
            time.sleep(SLEEP_BETWEEN)
            continue

        raw_rows, filtered_rows, zip_bytes, records = result

        jsonl_bytes = 0
        for rec in records:
            line = json.dumps(rec, ensure_ascii=False) + "\n"
            out_f.write(line)
            jsonl_bytes += len(line.encode("utf-8"))

        total_raw_rows      += raw_rows
        total_filtered_rows += filtered_rows
        total_zip_bytes     += zip_bytes
        total_jsonl_bytes   += jsonl_bytes
        days_collected      += 1

        print(
            f"{d} | raw={raw_rows:,} | filtered={filtered_rows:,} "
            f"| zip={zip_bytes/1e6:.1f}MB | jsonl={jsonl_bytes/1e6:.2f}MB"
        )
        time.sleep(SLEEP_BETWEEN)

print(f"\nDone. {days_collected} days collected, {days_skipped} skipped.")

In [ ]:
# ── Size report ────────────────────────────────────────────────────────────────
from datetime import date as _date

backfill_days = (_date.today() - _date(2021, 1, 1)).days

if days_collected > 0:
    avg_raw      = total_raw_rows      / days_collected
    avg_filtered = total_filtered_rows / days_collected
    avg_zip_mb   = total_zip_bytes     / days_collected / 1e6
    avg_jsonl_mb = total_jsonl_bytes   / days_collected / 1e6
    filter_pct   = 100 * total_filtered_rows / max(total_raw_rows, 1)

    ext_jsonl_gb = avg_jsonl_mb * backfill_days / 1e3
    ext_zip_gb   = avg_zip_mb   * backfill_days / 1e3
else:
    avg_raw = avg_filtered = avg_zip_mb = avg_jsonl_mb = filter_pct = 0
    ext_jsonl_gb = ext_zip_gb = 0

print("=" * 52)
print("  GDELT Events Sample Report")
print("=" * 52)
print(f"  Sample period     : {days[0]} → {days[-1]}  ({SAMPLE_DAYS} days)")
print(f"  Days collected    : {days_collected} / {SAMPLE_DAYS}")
print()
print("  Per-day averages:")
print(f"    Raw rows        : {avg_raw:,.0f}")
print(f"    Filtered rows   : {avg_filtered:,.0f}  ({filter_pct:.1f}% of raw)")
print(f"    Raw zip size    : {avg_zip_mb:.1f} MB")
print(f"    Bronze JSONL    : {avg_jsonl_mb:.2f} MB")
print()
print(f"  Extrapolated to full backfill (2021-01-01 → today, ~{backfill_days:,} days):")
print(f"    Bronze JSONL    : ~{ext_jsonl_gb:.1f} GB")
print(f"    Raw download    : ~{ext_zip_gb:.1f} GB")
print()
print(f"  Sample file saved : {output_path}")
print(f"  Total records     : {total_filtered_rows:,}")
print("=" * 52)

In [ ]:
# ── Preview ────────────────────────────────────────────────────────────────────
preview_cols = [
    "event_date", "actor1_country_code", "actor2_country_code",
    "quad_class", "goldstein_scale", "event_root_code",
    "num_sources", "avg_tone",
]

records_preview = []
with output_path.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 20:
            break
        records_preview.append(json.loads(line))

pd.DataFrame(records_preview)[preview_cols]

In [ ]:
# ── Zone-pair distribution ─────────────────────────────────────────────────────
pair_counts = {}
with output_path.open(encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        z1 = COUNTRY_TO_ZONE.get(rec.get("actor1_country_code"), "OTHER")
        z2 = COUNTRY_TO_ZONE.get(rec.get("actor2_country_code"), "OTHER")
        key = f"{z1} → {z2}"
        pair_counts[key] = pair_counts.get(key, 0) + 1

top_pairs = sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)[:15]
labels, values = zip(*top_pairs) if top_pairs else ([], [])

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(reversed(labels)), list(reversed(values)), color="steelblue")
ax.set_xlabel("Record count")
ax.set_title(f"Top zone pairs — {SAMPLE_DAYS}-day sample")
plt.tight_layout()
plt.show()

In [ ]:
# ── QuadClass distribution ─────────────────────────────────────────────────────
QUAD_LABELS = {
    2: "Material\nCooperation",
    3: "Verbal\nConflict",
    4: "Material\nConflict",
}
quad_counts = {2: 0, 3: 0, 4: 0}
with output_path.open(encoding="utf-8") as f:
    for line in f:
        q = json.loads(line).get("quad_class")
        if q in quad_counts:
            quad_counts[q] += 1

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    [QUAD_LABELS[q] for q in sorted(quad_counts)],
    [quad_counts[q] for q in sorted(quad_counts)],
    color=["#4CAF50", "#FF9800", "#F44336"],
)
ax.set_ylabel("Record count")
ax.set_title(f"QuadClass distribution — {SAMPLE_DAYS}-day sample")
plt.tight_layout()
plt.show()